In [3]:
from Bio import Entrez
import xmltodict
import urllib.request
from urllib.request import urlopen
import requests
from bs4 import BeautifulSoup as bs
import pandas as pd
import re
import time
import warnings
warnings.filterwarnings('ignore')

/tmp/ipykernel_546971/3624327311.py:7: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [6]:
"""
This function retrieves data on genes from the National Institutes of Health (NIH) website based on ACMG classification.

Args:
    gene_names_to_match: A list of gene names to search for.

Returns:
    A pandas DataFrame containing information about genes, including their corresponding MedGen UIDs and disease characteristics.
"""
def ACMG_dataextraction(gene_names_to_match):
    # Set the URL for the webpage containing the ACMG classification data
    url = "https://www.ncbi.nlm.nih.gov/clinvar/docs/acmg/"
    # Use requests library to fetch the content from the URL
    response = requests.get(url)
    # Check if the request was successful (status code 200 indicates success)
    if response.status_code == 200:
        # Parse the HTML content using BeautifulSoup
        soup = bs(response.text, 'html.parser')
        # Create a pandas DataFrame to store the extracted data
        data = pd.DataFrame(columns=['Concept_ID', 'Gene_Name'])

        # Extract all table elements from the webpage
        table_rows = soup.select('table')
        rows_to_concat = []  # Create an empty list to hold rows
        for row in table_rows:
            # Extract all table data (td) elements from the current row
            td_elements = row.find_all('td')
            # Iterate through the td elements in steps of 4 (assuming a specific data pattern in the table)
            for i in range(0, len(td_elements), 4):
                # Extract disease name from the current td element
                disease_name = td_elements[i].get_text(strip=True)
                medgen_link = td_elements[i + 1].find('a', href=lambda value: value and "/medgen/" in value)
                gene_link = td_elements[i + 2].find('a', href=lambda value: value and ("/gene/" in value or "/genes/" in value))
                
                # Check if both MedGen and gene links are found
                if medgen_link and gene_link:
                    medgen_id = medgen_link['href'].split('/')[-1]
                    medgen_id_cleaned = re.sub(r'[^a-zA-Z0-9]', '', medgen_id)
                    medgen_id_cleaned = re.sub(r'^term', '', medgen_id_cleaned)

                    gene_name = gene_link.get_text(strip=True)
                    row_data = {'Concept_ID': medgen_id_cleaned, 'Gene_Name': gene_name}
                    rows_to_concat.append(row_data)  # Append row data to list
                    
        # Concatenate all the rows collected earlier into the DataFrame
        data = pd.concat([data] + [pd.DataFrame(rows_to_concat)])  # Concatenate all rows
        
        # Get the minimum length between the Concept_ID and Gene_Name columns (to avoid index mismatch)
        min_length = min(len(data['Concept_ID']), len(data['Gene_Name']))
        data['Concept_ID'] = data['Concept_ID'][:min_length]
        data['Gene_Name'] = data['Gene_Name'][:min_length]

        MedgenUID_df = pd.DataFrame(data)
        # Loop through each row in the DataFrame
        for index, row in MedgenUID_df.iterrows():
            if 'OR' in row['Concept_ID']:
                split_ids = row['Concept_ID'].split('OR')
                MedgenUID_df.at[index, 'Concept_ID'] = split_ids[0].strip()
                new_row = row.copy()
                new_row['Concept_ID'] = split_ids[1].strip()
                new_row_df = pd.DataFrame([new_row])  # Convert the row to a DataFrame
                MedgenUID_df = pd.concat([MedgenUID_df, new_row_df], ignore_index=True)
                
        # Filter the DataFrame to rows where Gene_Name matches the gene_names_to_match list
        MedgenUID_df = MedgenUID_df[MedgenUID_df['Gene_Name'].isin([gene_names_to_match])].copy()
        MedgenUID_df['Medgen_UID'] = None
        # Loop through each row in the DataFrame again
        for index, row in MedgenUID_df.iterrows():
            # Build the URL to search for the MedGen ID using EUtils
            search_url = f'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=medgen&term={row["Concept_ID"]}&retmode=xml'
            time.sleep(2)
            search_response = requests.get(search_url)
            search_data = xmltodict.parse(search_response.text)
            # Check if the search returned a list of IDs and it's not None
            if 'IdList' in search_data['eSearchResult'] and search_data['eSearchResult']['IdList'] is not None:
                # Extract the IDs from the search response
                ids = search_data['eSearchResult']['IdList']['Id']
                # Update the 'Medgen_UID' column in the DataFrame with the IDs 
                MedgenUID_df.at[index, 'Medgen_UID'] = ids
        MedgenUID_df = MedgenUID_df.explode('Medgen_UID')
        # Loop through each row
        for index, row in MedgenUID_df.iterrows():
            medgen_url = f'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi?db=medgen&id={row["Medgen_UID"]}&retmode=xml'
            time.sleep(2)
            medgen_response = requests.get(medgen_url)
            if medgen_response.status_code == 200:
                try:
                    medgen_data = xmltodict.parse(medgen_response.text)
                    # Extract the disease characteristics from the summary data 
                    disease_characteristics = medgen_data['eSummaryResult']['DocumentSummarySet']['DocumentSummary']['Definition']
                    # Update the 'disease_characteristics' column in the DataFrame with the extracted data
                    MedgenUID_df.at[index, 'disease_characteristics'] = disease_characteristics
                except KeyError as e:
                    # Handle cases where the 'Definition' key might not be present
                    disease_characteristics = None
            else:
                # Print an error message if fetching the summary information fails
                print(f"Error: Unable to fetch data for ID {row['Medgen_UID']}. Status code: {medgen_response.status_code}")
    else:
        print("Failed to retrieve the page. Status code:", response.status_code)
    #print(MedgenUID_df)
    return MedgenUID_df
#ACMG_dataextraction('TP53')